# 编写Tokenizer

Tokenizer（分词器）是 NLP 中的“第一道工序”，负责把原始文本转换成模型能“看懂”的离散符号序列。  
核心作用一句话：把字符串 → 序列 of 整数（token id）。
1. 粒度层级  
   - 字符级：每个 Unicode 字符一个 token，词典小但序列长。  
   - 词级：按空格/标点切词，词典爆炸（几十万以上），且无法处理 OOV。  
   - 子词级（BPE、WordPiece、Unigram、SentencePiece）：高频词整体保留，低频词拆成高频子串，平衡词典大小与语义粒度，是当前主流。
2. 子词算法速览  
   BPE：从字符集合开始，统计相邻 pair 频次，迭代合并最高频 pair，直到达到预设词典大小。  
   WordPiece：类似 BPE，但用似然增益而非频次决定是否合并。  
   Unigram：先做大词典，再逐步删除使总似然下降最小的子词，用 EM 估计概率。  
   SentencePiece：把空格也当普通符号，先转 ▁，再做 BPE/Unigram，实现“无预分词”端到端。
3. 编码与解码  
   编码：文本 → tokenizer → token id list（可能加特殊符号：CLS、SEP、MASK 等）。  
   解码：token id list → tokenizer → 文本（合并子词、去掉特殊符号、空格还原）。
4. 额外能力  
   - 自动添加前缀/后缀空格控制；  
   - 提供 attention mask、token type id；  
   - 支持 fast 版本（Rust 后端）批量加速；  
   - 可扩展：添加新 token 只需在词典插入一行，再 resize 模型 embedding 矩阵即可。
5. 一句话总结  
   Tokenizer 决定了模型“第一眼”看到的世界：切得太粗会丢字，切得太细会丢意，子词在“词典大小、压缩率、语义完整性”之间找到了甜点区。

In [1]:
import re  # 引入正则表达式模块，用于实现一个简洁的分词器
from typing import List  # 引入类型注解：List 类型

def word_tokenize(text: str) -> List[str]:  # 定义一个简单分词函数，输入文本，输出 token 列表
    # 简单分词：按“词符/标点”切分，保留标点作为独立 token
    # 正则含义：\w+ 匹配词符（字母/数字/下划线）；[^\w\s] 匹配非词符且非空白（即标点）
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)  # 使用 UNICODE 模式适配中英混合

class Tokenizer:  # 定义一个最简 Tokenizer，实现文本到 id 序列的编码
    def __init__(self, word2idx: dict, idx2word: dict, max_len: int | None = None, add_bos: bool = False, add_eos: bool = False):  # 初始化方法
        """
        初始化 Tokenizer 类
        :param word2idx: 单词到索引的映射字典
        :param idx2word: 索引到单词的映射字典
        :param max_len: 最大序列长度，None 表示按批次最大长度
        :param add_bos: 是否在序列开头添加 <bos>
        :param add_eos: 是否在序列末尾添加 <eos>
        """
        self.word2idx = word2idx  # 词 -> id 字典
        self.idx2word = idx2word  # id -> 词 字典
        self.max_len = max_len  # 最大序列长度（None 表示动态取本批次最大）
        # 特殊 token
        self.pad_token = "<pad>"  # 填充 token
        self.unk_token = "<unk>"  # 未登录词 token
        self.bos_token = "<bos>"  # 序列起始 token
        self.eos_token = "<eos>"  # 序列结束 token
        self.add_bos = add_bos  # 是否默认添加 <bos>
        self.add_eos = add_eos  # 是否默认添加 <eos>
        # 特殊 token id（若词表未提供则给出默认值）
        self.pad_id = self.word2idx.get(self.pad_token, 0)  # pad 的 id，默认为 0
        self.unk_id = self.word2idx.get(self.unk_token, 1)  # unk 的 id，默认为 1
        self.bos_id = self.word2idx.get(self.bos_token, 2)  # bos 的 id，默认为 2
        self.eos_id = self.word2idx.get(self.eos_token, 3)  # eos 的 id，默认为 3

    def encode(self, texts: List[str], add_bos: bool | None = None, add_eos: bool | None = None) -> List[List[int]]:  # 批量编码接口
        """
        将文本批量编码为 token id 序列，并按批次进行 padding/截断。
        规则：
        - pad 长度 = min(本批次最大长度, self.max_len 或 +∞)
        - add_bos/add_eos 优先使用方法参数，否则使用类属性
        """
        if add_bos is None:  # 若未传入，回退到类级默认
            add_bos = self.add_bos  # 使用构造函数中的设置
        if add_eos is None:  # 若未传入，回退到类级默认
            add_eos = self.add_eos  # 使用构造函数中的设置

        batch_ids: List[List[int]] = []  # 存放每条样本的 id 序列
        for text in texts:  # 遍历每条文本
            tokens = word_tokenize(text)  # 分词得到 token 列表
            ids = [self.word2idx.get(tok, self.unk_id) for tok in tokens]  # 将 token 映射到 id，OOV 映射为 unk_id
            if add_bos:  # 如需在开头添加 <bos>
                ids = [self.bos_id] + ids  # 在序列最前添加 bos id
            if add_eos:  # 如需在末尾添加 <eos>
                ids = ids + [self.eos_id]  # 在序列最后添加 eos id
            batch_ids.append(ids)  # 收集本条样本的 id 序列

        max_len_in_batch = max(len(x) for x in batch_ids) if batch_ids else 0  # 计算本批次的最大序列长度
        pad_len = max_len_in_batch if self.max_len is None else min(max_len_in_batch, self.max_len)  # 取批次最大或 self.max_len 中较小者

        padded_ids: List[List[int]] = []  # 存放 padding/截断后的 id 序列
        for ids in batch_ids:  # 遍历每条样本
            if len(ids) < pad_len:  # 若长度不足 pad_len
                ids = ids + [self.pad_id] * (pad_len - len(ids))  # 在末尾补 pad，直到等长
            else:  # 若长度超出 pad_len
                ids = ids[:pad_len]  # 直接截断到 pad_len
            padded_ids.append(ids)  # 收集结果
        return padded_ids  # 返回二维 id 列表（形状：batch_size x pad_len）


In [ ]:
vocab = ["<pad>", "<unk>", "<bos>", "<eos>", "我", "爱", "NLP", "。"]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

tok = Tokenizer(word2idx, idx2word, max_len=8, add_bos=True, add_eos=True)
tok.encode(["我爱NLP。", "我 爱 NLP 。"])


'\n[[2, 1, 7, 3, 0, 0], [2, 4, 5, 6, 7, 3]]\n\n'

[[2, 1, 7, 3, 0, 0], [2, 4, 5, 6, 7, 3]]
- 第一个句子是"我爱NLP。"，编码为[2, 1, 7, 3, 0, 0]
- 第二个句子是"我 爱 NLP 。"，编码为[2, 4, 5, 6, 7, 3]
- 其中，2表示"<bos>"，3表示"<eos>"，0表示"<pad>"，1表示"<unk>"，4表示"我"，5表示"爱"，6表示"NLP"，7表示"。"
- 可以看到，两个句子的编码长度都是6，因为我们设置了max_len=8，且每个句子都添加了"<bos>"和"<eos>"
- 第一个句子中，"我"和"爱"的索引分别是4和5，"NLP"的索引是6，"。"的索引是7
- 第二个句子中，"我"和"爱"的索引分别是4和5，"NLP"的索引是6，"。"的索引是7